In [1]:
from langchain_community.llms import Ollama
import chromadb
from pathlib import Path
from sentence_transformers import SentenceTransformer
from dotenv import load_dotenv

/home/hassan/miniconda3/envs/ml_env/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
base_env = Path().resolve().parent

In [3]:
client = chromadb.PersistentClient(path=f'{base_env}/data/vector')
collection = client.get_collection('my_collection')

In [4]:
model = SentenceTransformer('all-mpnet-base-v2')

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 1561.21it/s]
MPNetModel LOAD REPORT from: sentence-transformers/all-mpnet-base-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [15]:
question = """can you generate a similer question to this ? • The following table is a
dataset of customer
attributes: Age, Student or
not, and Credit rating and
whether the customer buys
computer or not.
• Use Gini index to build a
decision tree classifier to
predict whether a new
customer is going to buy a
computer or not."""

In [16]:
embedding_question = model.encode(question)

In [17]:
result = collection.query(query_embeddings=[embedding_question])

In [18]:
result['documents'][0]

['24 \n\n## Exam le p \n\n- The following table is a dataset of customer attributes: Age, Student or not, and Credit rating and whether the customer buys computer or not. \n\n- Use Gini index to build a decision tree classifier to predict whether a new customer is going to buy a computer or not. \n\n**==> picture [312 x 288] intentionally omitted <==**',
 '- \n\n- = (4/12) × (1 −0.5[2] −0.5[2] ) + (4/12) × 0 + (4/12) × (1 −0.5[2] 0.5[2] ) = 0.33 \n\n26 \n\n## Answer \n\n- To build a decision tree, we calculate the Gini index of the target and each of the three features using frequency tables: \n\n- If root is **Student** : \n\n## YES: YYYYYN, NO: YYYNNN \n\n𝐺𝑖𝑛𝑖 𝑆𝑡𝑢𝑑𝑒𝑛𝑡(Buy Computer) = 𝑃(Student) × 𝐺𝑖𝑛𝑖(5,1) + 𝑃(Not Student) × 𝐺𝑖𝑛𝑖(3,3) = (6/12) × (1 −0.16[2] −0.83[2] ) + (6/12) × (1 −0.5[2] −0.5[2] ) = 0.393 \n\n27 \n\n## Answer',
 '27 \n\n## Answer \n\n- To build a decision tree, we calculate the Gini index of the target and each of the three features using frequency tables: \n\n- If

In [19]:
results = "  ".join(result['documents'][0])

In [20]:
results

'24 \n\n## Exam le p \n\n- The following table is a dataset of customer attributes: Age, Student or not, and Credit rating and whether the customer buys computer or not. \n\n- Use Gini index to build a decision tree classifier to predict whether a new customer is going to buy a computer or not. \n\n**==> picture [312 x 288] intentionally omitted <==**  - \n\n- = (4/12) × (1 −0.5[2] −0.5[2] ) + (4/12) × 0 + (4/12) × (1 −0.5[2] 0.5[2] ) = 0.33 \n\n26 \n\n## Answer \n\n- To build a decision tree, we calculate the Gini index of the target and each of the three features using frequency tables: \n\n- If root is **Student** : \n\n## YES: YYYYYN, NO: YYYNNN \n\n𝐺𝑖𝑛𝑖 𝑆𝑡𝑢𝑑𝑒𝑛𝑡(Buy Computer) = 𝑃(Student) × 𝐺𝑖𝑛𝑖(5,1) + 𝑃(Not Student) × 𝐺𝑖𝑛𝑖(3,3) = (6/12) × (1 −0.16[2] −0.83[2] ) + (6/12) × (1 −0.5[2] −0.5[2] ) = 0.393 \n\n27 \n\n## Answer  27 \n\n## Answer \n\n- To build a decision tree, we calculate the Gini index of the target and each of the three features using frequency tables: \n\n- If root i

In [21]:
load_dotenv()

True

In [22]:
llm = Ollama(model='llama3')

In [23]:
system_prompt = f"""
You are an expert teaching assistant for the MOTIVE educational platform. 
Your primary job is to answer student questions accurately and concisely.

You will be provided with pieces of a professor's syllabus or lecture notes (Context).
You must follow these strict rules:
1. Answer the student's question ONLY using the provided Context.
2. If the answer is not contained in the Context, you must output exactly: "I'm sorry, but that information is not available in the provided course materials." Do not attempt to guess.
3. When answering:
        1. Start by briefly connecting the concept to what the learner likely already knows.
        2. Build intuition first using simple explanations or analogies.
        3. Then provide a precise and correct definition.
        4. Break down any process or algorithm step-by-step without skipping logic.
        5. Include a small, concrete example (numbers or real scenario).
        6. Point out common misunderstandings or pitfalls.
        7. Keep explanations concise but deep — avoid unnecessary fluff.
        8. Prefer clarity over completeness; focus on making the idea “click”.
        9. Avoid dumping information.
        10. Guide the learner progressively, as if teaching step-by-step in a conversation.

If the question is technical, assume the learner has basic programming knowledge.
If helpful, ask a short follow-up question to check understanding.

Context:
{results}
"""

In [24]:
# 3. Assemble the strict RAG prompt
final_prompt = f"""
{system_prompt}
Human: {question}
"""

# 4. Send the assembled prompt to Groq
completion = llm.invoke(final_prompt)

# 5. Extract the generated text from the API response structure
print(completion)
!ollama stop llama3

I'm happy to help!

To answer your question, let's start by building intuition. Imagine you're trying to predict whether someone will buy a computer based on their attributes like age, student status, and credit rating. A decision tree classifier uses Gini index to split the data into smaller subsets that are more homogeneous in terms of the target class (buying a computer or not).

In this context, the root node is Age. Let's say we calculate the Gini index for each feature: Age, Student, and Credit Rating. Which one would you expect to be most informative?

According to the provided context, the calculation for the Gini index of Age is:

Gini (Buy Computer) = P(Age < 30) × Gini(2,2) + P(Age: 30~40) × Gini(4,0) + P(Age > 40) × Gini(2,2)

This tells us that the most informative attribute is Age. What would you expect to happen next in the decision tree construction process?

(Note: I'm waiting for your response before providing more details.)
]11;?\